In [1]:
import torch
from torch.utils.data import DataLoader
import torch.optim as optim
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from torch.utils.data import Dataset
import torch.nn as nn
import torch.nn.functional as F

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# Load the dataset
df = pd.read_csv('/content/drive/MyDrive/202509_CURRENT_1024x1024dataset.csv', index_col=0)

# Parse columns
unique_region = df["unique_region"].values
labels = df["Cell Type"].values

# Cell positions
x_positions = df['x'].to_numpy()
y_positions = df['y'].to_numpy()
print(x_positions)

# Drop everything not used in GNN features
drop_cols = [
    "Cell Type", "unique_region", "donor", "array", "Xcorr", "Ycorr",
    "Tissue_location", "tissue", "region", "OLFM4", "FAP", "CD25",
    "CK7", "MUC6", "Cell Type em", "Cell subtype", "Neighborhood",
    "Neigh_sub", "Neighborhood_Ind", "NeighInd_sub", "Community",
    "Major Community", "Tissue Segment", "Tissue Unit", "CollIV"
]
keep_cols = ['CD34', 'CD38', 'Cytokeratin', 'CD19', 'CD4', 'CD49f', 'aSMA', 'CD161', 'CD163', 'NKG2D', 'CD16', 'CD49a', 'CD138',  'CD8', 'CD206',  'CD117', 'CD36', 'CD7', 'SOX9', 'CD68', 'CD57', 'Podoplanin', 'CD44', 'CD56', 'CD31', 'MUC1', 'aDef5', 'CD3', 'Synapto', 'ITLN1', 'Vimentin', 'CD15', 'CD11c', 'CD45', 'CD66', 'HLADR', 'Ki67', 'CD21',  'CD90', 'CHGA', 'CD123']
features_df = df[keep_cols]
feature_cols = features_df.columns

/tmp/ipython-input-2092688328.py:2: DtypeWarning: Columns (73) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/202509_CURRENT_1024x1024dataset.csv', index_col=0)


[434 565 661 ... 723 227 232]


In [7]:
# Dataset fot mlp classification

class CellFeatureDataset(Dataset):
    def __init__(self, df, feature_cols, label_col):
        """
        df: pandas DataFrame, each row is a cell
        feature_cols: list of biomarker feature column names
        label_col: cell type label (already encoded)
        """
        self.features = df[feature_cols].values  # shape [num_cells, num_features]
        self.labels = df[label_col].values        # shape [num_cells]

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        x = torch.tensor(self.features[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


In [8]:
# mlp classifier model

class MLPClassifier(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)

        self.fc2 = nn.Linear(256, 1024)
        self.bn2 = nn.BatchNorm1d(1024)

        self.fc3 = nn.Linear(1024, 256)
        self.bn3 = nn.BatchNorm1d(256)


        self.fc4 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        x = F.relu(self.bn3(self.fc3(x)))
        x = self.fc4(x)
        return x


In [ ]:
# MLP Classification Entrance

# Encode labels
label_encoder = LabelEncoder()
df['Cell Type'] = label_encoder.fit_transform(df['Cell Type'])

# Split train/test
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["Cell Type"]
)

# Create datasets
train_dataset = CellFeatureDataset(train_df, feature_cols, label_col="Cell Type")
test_dataset = CellFeatureDataset(test_df, feature_cols, label_col="Cell Type")

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --------- Define model ----------
model = MLPClassifier(
    in_dim=len(feature_cols),
    hidden_dim=512,
    num_classes=len(np.unique(df['Cell Type']))
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

# --------- Training ----------
for epoch in range(1, 20):
    model.train()
    total_loss = 0
    correct = 0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        out = model(x_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = out.argmax(dim=1)
        correct += (preds == y_batch).sum().item()

    acc = correct / len(train_dataset)
    print(f"Epoch {epoch}, Loss {total_loss/len(train_dataset):.10f}, Train Acc {acc:.10f}")

# --------- Evaluate on Test Set ----------
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        outputs = model(x_batch)
        preds = outputs.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

# Compute accuracy
test_acc = np.mean(np.array(all_preds) == np.array(all_labels))
print(f"Test Accuracy: {test_acc:.4f}")